# HAB-P15 on the GPU: brain and feelings co-evolve, in short and long lives

40 simulated **Unitree Go2** robots. Each has its own brain, which **learns to walk** (PPO) under its
own **feelings** about its body: how much motor heat bothers it, what effort costs, whether it enjoys
moving. Both are inherited: each round, in every group of 4, the worst robot takes a copy of the best
robot's brain and a mutated copy of its feelings.

There are two worlds that differ only in how long their robots live: **4** or **32** one-minute missions,
with permanent heat damage. They start from identical brains and feelings. At the end comes a
**reciprocal transplant**: is each population best in its own world?

Pre-registration: [`docs/prereg/HABP15_coevolution.md`](https://github.com/pablopedrosap/homeostatic-Alife/blob/main/docs/prereg/HABP15_coevolution.md)

**How to run:** Session options → **GPU T4 x2**, **Internet on** → **Save Version → Save & Run All**
(~8–11 h; you can close the tab). Results: the version's **Output** tab → `results/habp15_gpu/`.

In [ ]:
# 1) Setup — exact versions tested together (do not change them)
import os
os.environ["MUJOCO_GL"] = "egl"                       # headless rendering on the GPU server
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false" # leave GPU memory for the video renderer
!pip install -q "jax[cuda12]==0.9.2" "brax==0.14.2" "mujoco==4.0.0" "mujoco-mjx==4.0.0" mediapy

In [ ]:
# 2) Check that JAX sees the GPU
import jax
print(jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU! Kaggle: right panel → Session options → Accelerator → GPU T4 x2"


In [ ]:
# 3) Get the project code and the Go2 robot model
import sys, warnings
warnings.simplefilter("ignore")
!rm -rf homeostatic-Alife && git clone -q --depth 1 https://github.com/pablopedrosap/homeostatic-Alife.git
sys.path.insert(0, "homeostatic-Alife/kaggle")
import go2_body_budget as G
xml = G.fetch_go2("mujoco_menagerie")
OUT = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
os.makedirs(OUT, exist_ok=True)
print("robot:", xml, "| results go to:", OUT)

## The run
12 rounds × 1M PPO steps per robot. If time runs short, it stops evolving early and still runs the
final test (the report states how many rounds were completed). Progress lines show each world's mean
walking speed.

In [ ]:
import coevolution as C
cfg = C.CoConfig()      # 2 worlds x 5 groups x 4 robots; stops evolving at ~11 h to leave time for the final test
co = C.CoEvolution(xml, cfg, out_dir=f"{OUT}/habp15_gpu")
co.run()

## Pre-registered analysis

In [ ]:
!python homeostatic-Alife/scripts/analyze_go2_v15.py --dir {OUT}/habp15_gpu --fig-dir {OUT}/habp15_gpu
from IPython.display import Image
Image(f"{OUT}/habp15_gpu/habp15_coevolution.png")

## Video: the best robot of each world, carrying 3 kg (new body)

In [ ]:
import mediapy
last = max(h["round"] for h in co.history)
for w in ("short", "long"):
    best = max((h for h in co.history if h["round"] == last and h["world"] == w), key=lambda h: h["fitness"])
    frames = G.render(xml, co.record(best["id"], payload_kg=3.0, seconds=20.0))
    mediapy.write_video(f"{OUT}/habp15_gpu/{w}_world_best.mp4", frames, fps=25)
    print(w, "world, robot", best["id"]); mediapy.show_video(frames, fps=25)

## Results
Download `results/habp15_gpu/` from the Output tab and bring it back to the repository
(`history.json`, `transplant.json`, `run_config.json`, `verdict.txt`, the figure and the videos).